# Capitulo 13 - Pipeline RAG Completo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap13_rag_pipeline.ipynb)

Neste capitulo, construimos um pipeline RAG (Retrieval-Augmented Generation) completo do zero.
Cobrimos desde os fundamentos ate a integracao com bancos de dados vetoriais.

---

## Atribuicao e Fonte Original

**Fonte:** [RAG-with-Python-Cookbook](https://github.com/PacktPublishing/RAG-with-Python-Cookbook) - Packt Publishing

**Notebooks fonte:** ch01_RAG_intro/rag_basics.ipynb, ch02_generation/generation.ipynb, ch05_text_embedding/text_embeddings.ipynb, ch06_similarity_search_vector_databases/vector_databases.ipynb

**Adaptado e comentado por Anderson Ejepsen** para o livro *LLM On-Premise: Guia Pratico*.

Todos os creditos aos autores originais sao mantidos.

---

## Instalacao de Dependencias

Execute a celula abaixo para instalar todos os pacotes necessarios para este capitulo.

In [ ]:
# Instalar todas as dependencias do capitulo
%pip install -q numpy openai chromadb tiktoken python-dotenv \
    sentence-transformers faiss-cpu anthropic httpx requests matplotlib

---
## Parte 1: Fundamentos do RAG

Nesta secao, exploramos os conceitos basicos de Retrieval-Augmented Generation (RAG).
O RAG combina recuperacao de informacao com geracao de texto para produzir respostas fundamentadas em dados reais.

## Instalar Pacotes Necessarios

In [ ]:
%pip install "numpy<2" openai==1.12.0 chromadb==0.4.22 tiktoken==0.5.2 python-dotenv==1.0.0

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Verificar if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Carregar arquivos de exemplo

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata  # type: ignore

    api_key = userdata.get("OPENAI_API_KEY")

    if not api_key:
        raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

    os.environ["OPENAI_API_KEY"] = api_key

## Step 1: Text Chunking

Split documents into smaller chunks with overlap to preserve context.

In [ ]:
from pathlib import Path
import chromadb
from openai import OpenAI


def chunk_text(text, size=1000, overlap=200):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            bp = text.rfind("\n\n", start, end)
            if bp == -1:
                bp = text.rfind(". ", start, end)
            if bp > start:
                end = bp + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end < len(text) else end
    return chunks


base_datasets_dir = Path("/content/datasets") if IN_COLAB else Path("../datasets")
file_path = base_datasets_dir / "text_files" / "harry_potter_knowledge_base.txt"
if not file_path.exists():
    file_path = Path("./datasets/text_files/harry_potter_knowledge_base.txt")

text = file_path.read_text(encoding="utf-8")
chunks = chunk_text(text)

## Step 2: Embed and Store in ChromaDB

Generate embeddings and store them in a vector database.

In [ ]:
import httpx # Add this import
client = OpenAI(http_client=httpx.Client())
embedding_model = "text-embedding-3-small"


def embed_and_store(chunks, db_path, collection_name):
    chroma = chromadb.PersistentClient(path=str(db_path))
    collection = chroma.get_or_create_collection(
        name=collection_name,
        metadata={"description": "Harry Potter knowledge base"},
    )

    embeddings = []
    for i in range(0, len(chunks), 100):
        batch = chunks[i : i + 100]
        res = client.embeddings.create(model=embedding_model, input=batch)
        embeddings.extend([x.embedding for x in res.data])

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embeddings,
        metadatas=[{"chunk_index": i} for i in range(len(chunks))],
    )
    return collection


chroma_db_dir = Path("chroma_db")
collection = embed_and_store(chunks, chroma_db_dir, "harry_potter_kb")
collection

## Step 3: Test Retrieval

Test that the retrieval system finds relevant chunks for a given question.

In [ ]:
def retrieve(question, top_k=3):
    q_emb = client.embeddings.create(
        model=embedding_model,
        input=question,
    ).data[0].embedding

    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents"],
    )

    return res["documents"][0]


question = "Why did Uncle Vernon take the family to a hut in the middle of the sea?"
docs = retrieve(question)
docs

## Step 4: Test Generation

Feed the retrieved documents and question to the LLM to generate an answer.

In [ ]:
def answer(question, docs):
    context = "\n\n---\n\n".join(docs)
    prompt = f"""Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:"""

    res = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
    )

    return res.choices[0].message.content


question = "Why did Uncle Vernon take the family to a hut in the middle of the sea?"
docs = retrieve(question)
answer_text = answer(question, docs)
answer_text

---
## Parte 2: Modelos de Geracao

Exploramos diferentes modelos de geracao (OpenAI, Anthropic, modelos locais) e como integra-los ao pipeline RAG.

**Instalacao de dependencias:** A celula abaixo instala os pacotes Python necessarios.

In [ ]:
%pip install openai==2.21.0 anthropic==0.18.1 python-dotenv==1.0.0 httpx==0.27.0

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Verificar if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Carregar arquivos de exemplo

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

## 1. OpenAI Chat Completions

In [ ]:
from openai import OpenAI

def ask_with_context(context, question):
    client = OpenAI()

    messages = [
        {
            "role": "system",
            "content": "Answer based only on the provided context."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}"
        },
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    return response.choices[0].message.content


# Usage
context = "RAG stands for Retrieval-Augmented Generation."
question = "What does RAG stand for?"
answer = ask_with_context(context, question)
print(answer)

## 2. OpenAI Whisper Speech-to-Text

In [ ]:
import os
os.listdir()

In [ ]:
from openai import OpenAI
import os

client = OpenAI()

# Determine the correct path based on environment
if IN_COLAB:
    audio_path = "/content/datasets/audio_files/harvard.wav"
else:
    audio_path = "../datasets/audio_files/harvard.wav"

with open(audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
    )

print(transcript.text)

## 3. Anthropic Claude Example

In [ ]:
client.models.list()

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Explain how vector databases work in "
                       "simple terms.",
        }
    ],
)

print(response.content[0].text)

## 4. Gemini API Example using the OpenAI SDK

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

client.models.list()

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

resp = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
)

print(resp.choices[0].message.content)

## 5. Deploy local LLMs using Ollama

### Setting up Ollama in Colab

Since Colab runs in a cloud environment, you need to install and run Ollama directly within the Colab instance. The following steps will set up Ollama and pull the required models.

In [ ]:
import subprocess
import os

# Instalar zstd dependency
!sudo apt-get update && sudo apt-get install -y zstd

# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar Ollama server in the background
os.environ['OLLAMA_HOST'] = '0.0.0.0'
subprocess.Popen(["ollama", "serve"])

print("Ollama server is starting...")

In [ ]:
# Give Ollama some time to ensure it's fully started (adjust if needed)
import time
time.sleep(10)

print("Pulling models...")
!ollama pull qwen3:4b
!ollama pull llama2
!ollama pull mistral
!ollama pull codellama

print("Models pulled successfully. You can now re-run the Ollama examples.")

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
from openai import OpenAI

# Point the client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama does not require a real key,
                       # but the SDK expects one
)

response = client.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is retrieval augmented generation?"
        },
    ],
)

print(response.choices[0].message.content)

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
from openai import OpenAI

models = ["llama2", "mistral", "codellama"]
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

for model in models:
    print(f"\n--- Testing {model} ---")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": "Explain RAG in one sentence."}
        ]
    )
    print(response.choices[0].message.content)

## 6. Pydantic Structured Output

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
from datetime import date
from typing import List

class LineItem(BaseModel):
    description: str
    quantity: int
    total: float

class Invoice(BaseModel):
    invoice_number: str
    invoice_date: date
    supplier: str
    items: List[LineItem]
    total_due: float

client = OpenAI()

completion = client.chat.completions.parse(
    model="gpt-5",
    messages=[
        {"role": "system", "content": "Extract the invoice data from the provided context."},
        {"role": "user", "content": "Invoice #12345, Date: 2024-01-15, Supplier: Tech Corp. Item: Laptop, Qty: 2, Total: $2000. Item: Mouse, Qty: 5, Total: $100. Total Due: $2100"}
    ],
    response_format=Invoice,
)

invoice = completion.choices[0].message.parsed
print(invoice)

---
## Parte 3: Embeddings de Texto

Embeddings sao representacoes vetoriais densas que capturam o significado semantico do texto.
Sao fundamentais para a busca por similaridade no RAG.

**Instalacao de dependencias:** A celula abaixo instala os pacotes Python necessarios.

In [ ]:
!pip install sentence-transformers==4.1.0
!pip install requests==2.32.3
!pip install matplotlib==3.10.3
!pip install openai==1.79.0
!pip install langchain-openai==0.3.17
!pip install pandas==2.2.3
!pip install rank-bm25
!pip install langchain_text_splitters
!pip install openai==1.83.0
!pip install PyPDF2==3.0.1

### Carregar arquivos de exemplo

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Verificar if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### How to generate embeddings using the OpenAI and HuggingFace API

In [ ]:
from openai import OpenAI

text_chunks = ["The sky is blue.", "The grass is green."]

client = OpenAI()  # uses the environment variable OPENAI_API_KEY

embeddings_list = []

for text_chunk in text_chunks:
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    embeddings_list.append(embedding)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings_langchain_list = []

for text_chunk in text_chunks:
    embeddings_openai_langchain = embeddings.embed_query(text_chunk)
    embeddings_langchain_list.append(embeddings_openai_langchain)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
from sentence_transformers import SentenceTransformer

# Carregar the model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
text_chunks = ["The sky is blue.", "The grass is green."]

embeddings = model.encode(text_chunks)

### Calculating the Distance Between Embeddings

In [ ]:
import numpy as np
from numpy.linalg import norm
import pandas as pd
import os
from openai import OpenAI

text_chunks = [
    "The Great Fire of London in 1666 destroyed over 13,000 houses.",
    "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
    "The Black Death is estimated to have killed nearly one-third of the population.",
]

users_question = "Tell me something interesting about diseases in history"

embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])

client = OpenAI()
embeddings = []

def create_embeddings(text_chunk, client):
    embedding = (
        client.embeddings.create(input=[text_chunk], model="text-embedding-3-small")
        .data[0]
        .embedding
    )
    return embedding

# Apply function create_embeddings to the correct column
embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(
    create_embeddings, client=client
)

users_question_embedding = create_embeddings(
    text_chunk=users_question, client=client
)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
# create a list to store the calculated cosine similarity
cos_sim = []

def calculate_cosine_similarity(text_chunk_embedding, users_question_embedding):
    A = text_chunk_embedding
    B = users_question_embedding

    # calculate the cosine similarity
    cosine = np.dot(A, B) / (norm(A) * norm(B))
    return cosine

# Apply function calculate_cosine_similarity to the correct column
embeddings_df["similarity"] = embeddings_df["embedding"].apply(
    calculate_cosine_similarity, users_question_embedding=users_question_embedding
)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
from sentence_transformers.util import cos_sim

document_embeddings=embeddings_df["embedding"].tolist()
query_embedding=embeddings_df["embedding"].tolist()

for document_embedding in document_embeddings:
    # Computar cosine_similarity between documents and query
    scores = cos_sim(document_embedding, query_embedding)

scores

### How to reduce the dimensionality of embeddings to be able to plot them

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from openai import OpenAI

# Definir text chunks
text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

# Inicializar OpenAI client
# Assumes the OPENAI_API_KEY environment variable is set
client = OpenAI()
embeddings = []

# Gerar embeddings for text chunks
for text_chunk in text_chunks:
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embeddings.append(response.data[0].embedding)

# Converter embeddings to a DataFrame
# Each row represents one text chunk, each column one embedding dimension
embeddings_df = pd.DataFrame(
    embeddings, columns=[f"dim_{i}" for i in range(len(embeddings[0]))]
)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
# Perform PCA with 2 components
pca = PCA(n_components=2)
df_reduced = pca.fit_transform(embeddings_df)

# Criar a new DataFrame with reduced dimensions
df_reduced = pd.DataFrame(df_reduced, columns=["PC1", "PC2"])
df_reduced["text"] = text_chunks  # Add original text chunks for labeling

# Criar a scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(df_reduced["PC1"], df_reduced["PC2"])

# Adicionar labels to each point
for i, label in enumerate(df_reduced["text"]):
    plt.text(df_reduced["PC1"][i], df_reduced["PC2"][i], label, fontsize=9)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Scatter Plot")

# Salvar the plot
plt.savefig("principal_component_plot.svg", format="svg")

### How to calculate multimodal embeddings using CLIP

In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# Carregar the model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Text and image inputs
descriptions = ["A photo of a cat", "A photo of a dog"]
images = [
    Image.open("../datasets/images/cat.jpg"),
    Image.open("../datasets/images/dog.jpg"),
]

# Disable gradient tracking since this code only runs inference
with torch.no_grad():
    inputs = processor(
        text=descriptions,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    outputs = model(**inputs)

dot_products_per_text = outputs.logits_per_text

# Calcular probabilities
probabilities = dot_products_per_text.softmax(dim=1)

In [ ]:
print(probabilities)

print("Probability that the first image is a cat:", probabilities[0][0].item())
print("Probability that the second image is a dog:", probabilities[1][1].item())

### How to optimize similarity search by combining similarity search and keyword search

In [ ]:
from rank_bm25 import BM25Okapi
import pandas as pd

text_chunks = [
    "The Great Fire of London in 1666 destroyed over 13,000 houses.",
    "Julius Caesar was assassinated on the Ides of March (March 15) in 44 BCE.",
    "The Black Death is estimated to have killed nearly one-third of the European population.",
]

# Tokenize text into words
tokenized_chunks = [chunk.split(" ") for chunk in text_chunks]

bm25 = BM25Okapi(tokenized_chunks)

user_query = "Tell me something interesting about diseases in history"
tokenized_query = user_query.split(" ")

# BM25 scores for each document
bm25_scores = bm25.get_scores(tokenized_query)

# Document IDs ordered by keyword relevance (best first)
ranking_keyword_search = (
    pd.DataFrame(bm25_scores, columns=["score"])
    .sort_values(by="score", ascending=False)
    .index
    .to_list()
)
ranking_keyword_search


**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
from sentence_transformers import SentenceTransformer

embeddings_df = pd.DataFrame(text_chunks, columns=["text_chunk"])

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embeddings = []

def create_embeddings(text_chunk, client):
    embedding = (
        client.embeddings.create(input=[text_chunk], model="text-embedding-3-small")
        .data[0]
        .embedding
    )
    return embedding

embeddings_df["embedding"] = embeddings_df["text_chunk"].apply(
    create_embeddings, client=client
)

users_question_embedding = create_embeddings(text_chunk=user_query, client=client)

# Computar cosine_similarity between documents and query
embeddings_df["similarity"] = cos_sim(
    embeddings_df["embedding"], users_question_embedding
)

ranking_semantic_search = embeddings_df.sort_values(
    by=["similarity"], ascending=False
).index
ranking_semantic_search


In [ ]:
#  calculate a combined similarity score using Reciprocal Rank Fusion (RRF)
combined_score = []

for i in range(0, len(ranking_semantic_search), 1):
    k = 60
    rrf_score = 1 / (k + ranking_keyword_search[i]) + 1 / (
        k + ranking_semantic_search[i]
    )
    combined_score.append(rrf_score)

combined_score_df = pd.DataFrame(combined_score, columns=["combined_score"])
new_ranking = (
    combined_score_df.sort_values(by=["combined_score"], ascending=False).index + 1
)
new_ranking

### Performing Text Classification Using Embeddings

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2
import pandas as pd

pdf_files = [
    {
        "file_path": "../datasets/pdf_files/history_of_deep_learning.pdf",
        "label": "Deep_Learning",
    },
    {
        "file_path": "../datasets/pdf_files/premier_league_history.pdf",
        "label": "Premier_League",
    },
]

chunks_dict_list = []

# split both documents into chunks and append to the same list of dicts
for pdf_file in pdf_files:
    with open(pdf_file["file_path"], "rb") as file:
        reader = PyPDF2.PdfReader(file)

        text = ""
        for page in reader.pages:
            text += page.extract_text()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = text_splitter.split_text(text)

    for chunk in chunks:
        chunks_dict_list.append({"text": chunk, "label": pdf_file["label"]})

chunks_df = pd.DataFrame(chunks_dict_list)
chunks_df

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
client = OpenAI()

def create_embeddings(text_chunk, client):
    response = client.embeddings.create(
        input=[text_chunk], model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    return embedding

chunks_df["embedding"] = chunks_df["text"].apply(create_embeddings, client=client)
chunks_df.to_csv("chunks_df.csv")

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

y = chunks_df["label"]
X = chunks_df["embedding"].apply(
    lambda x: pd.Series(eval(x)) if isinstance(x, str) else pd.Series(x)
)

# Train a random forest classifier
clf = RandomForestClassifier()
clf.fit(X, y)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
test_embedding = create_embeddings(
    text_chunk="What is the name of the top football league in England?",
    client=client,
)

X_test = [test_embedding]

# Predict the most likely class
predicted_classes = clf.predict(X_test)  # e.g. ['Premier_League']
probabilities = clf.predict_proba(X_test)  # e.g. array([[0.06, 0.94]])
probabilities

---
## Parte 4: Busca por Similaridade e Bancos de Dados Vetoriais

Bancos vetoriais como ChromaDB e FAISS permitem busca eficiente por similaridade.
Sao a espinha dorsal do componente de recuperacao do RAG.

**Instalacao de dependencias:** A celula abaixo instala os pacotes Python necessarios.

In [ ]:
!pip install faiss-cpu==1.8.0
!pip install openai==2.14.0
!pip install chromadb==0.5.3
!pip install pandas==2.2.2
!pip install psycopg2==2.9.9

### Carregar arquivos de exemplo

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
else:
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Add it to Colab secrets or local environment variables.")

os.environ["OPENAI_API_KEY"] = api_key

### 1 Storing and Working with Embedding using FAISS

In [ ]:
import faiss
import numpy as np
from openai import OpenAI

# Exemplo list of sample strings
text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

# Inicializar the OpenAI embeddings model
client = OpenAI()
model = "text-embedding-3-small"

# Gerar embeddings for the sample strings
def get_embedding(text):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

document_embeddings = np.array(
    [get_embedding(text) for text in text_chunks]
)

# Converter embeddings to float32 (FAISS requires float32 type)
document_embeddings = document_embeddings.astype("float32")

# Criar a FAISS index (using L2 distance)
index = faiss.IndexFlatL2(document_embeddings.shape[1])

# Adicionar embeddings to the index
index.add(document_embeddings)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
# generate a query embedding for the user query
query = "What color are violets?"
query_embedding = np.array(get_embedding(query)).astype("float32")

# Perform the search: k = number of closest documents you want to retrieve
k = 5
distances, indices = index.search(query_embedding.reshape(1, -1), k)

# Recuperar the documents corresponding to the indices
retrieved_documents = [text_chunks[i] for i in indices[0]]

In [ ]:
print("Distances: ", distances)
print("Indices: ", indices)
print("Retrieved document: ", retrieved_documents)


#### 2 Storing and Working with Embeddings in ChromaDB

In [ ]:
import chromadb

# vector store settings
VECTOR_STORE_PATH = r"../02_Data/00_Vector_Store"
COLLECTION_NAME = "my_collection"

# get/create a chroma client and collection
chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)


from openai import OpenAI

text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

# Gerar embeddings
client = OpenAI()
model = "text-embedding-3-small"
def get_embedding(text):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

embeddings_model = client  # for compatibility with later code
embeddings_list = [get_embedding(text) for text in text_chunks]

# add data frame to collection
collection.add(
    embeddings=embeddings_list,
    documents=text_chunks,
    ids=[
        str(i) for i in range(len(text_chunks))
    ],  # create a list of strings as index
)


# query collection
query = "What is the color of the sky?"
query_embedding = get_embedding(query)
results = collection.query(query_embeddings=[query_embedding], n_results=3)

In [ ]:
results

#### 3 Storing and Working with Embeddings in a PostgreSQL database


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
)

cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

cur.execute(
    """
    CREATE TABLE IF NOT EXISTS embeddings (
        id SERIAL PRIMARY KEY,
        text TEXT,
        embedding VECTOR(1536)
    );
    """
)

conn.commit()
cur.close()
conn.close()

In [ ]:
conn

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
from openai import OpenAI

# Definir text chunks
text_chunks = [
    "The sky is blue.",
    "The sun is shining.",
    "I love chocolate.",
    "Ice cream is delicious.",
    "Roses are red.",
    "Violets are blue.",
]

client = OpenAI()
model = "text-embedding-3-small"

def get_embedding(text):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

index = 0
cur = conn.cursor()

# Insert the embeddings into the table
for text_chunk in text_chunks:
    embedding = get_embedding(text_chunk)
    cur.execute(
        """INSERT INTO embeddings
        (id, chunk, embedding)
        VALUES (%s, %s, %s)""",
        (index, text_chunk, embedding),
    )
    index += 1

In [ ]:
write_vectors_to_postgres(conn)

Query all data from table embeddings

In [ ]:
cur = conn.cursor()
cur.execute("SELECT * FROM embeddings_table")
rows = cur.fetchall()

for row in rows:
    print(row)

#### 4 Perform similarity search on top of the just create PostgreSQL table

In [ ]:
from openai import OpenAI

# Inicializar the OpenAI embeddings model
client = OpenAI()
model = "text-embedding-3-small"

# Exemplo text chunk
text_chunk = "Sweets are delicious."

# Obter the embedding
response = client.embeddings.create(input=text_chunk, model=model)
embeded_query = response.data[0].embedding

cur = conn.cursor()
cur.execute(
    f"""
    SELECT 1 - (embedding <=> '{embeded_query}') AS cosine_similarity, *
    FROM embeddings
    ORDER BY 1 - (embedding <=> '{embeded_query}') DESC
    LIMIT 20;
"""
)

results = cur.fetchall()

In [ ]:
results

### 5 Write a larger amount of data to the vector store

For this example we are using the job description dataset to write it to the database.

In [ ]:
import psycopg2
from psycopg2 import Error

conn = psycopg2.connect(
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
    host="localhost",
    port="5432",
)

cur = conn.cursor()
cur.execute("""CREATE EXTENSION IF NOT EXISTS vector""")

cur.execute(
    """
    CREATE TABLE IF NOT EXISTS job_description_table(
        id integer PRIMARY KEY,
        text_chunk TEXT,
        embedding vector(1536)
    )
    """
)
conn.commit()

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:

import psycopg2
from psycopg2 import Error
import pandas as pd

conn = psycopg2.connect(
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
    host="localhost",
    port="5432",
)

cur = conn.cursor()

# execute the query to create the table including the vector column
cur.execute(
    """
    CREATE TABLE IF NOT EXISTS job_description_table(
        id integer PRIMARY KEY,
        text_chunk TEXT,
        embedding vector(1536)
    )
    """
)
conn.commit()
from openai import OpenAI

file_path = "../datasets/large_text_dataset/resume_job_description_data.csv"
df = pd.read_csv(file_path)
text_chunks = df["job_description_text"]
print(len(text_chunks))

try:
    # get all entries from the table
    cur.execute("SELECT * FROM job_description_table")
    rows = cur.fetchall()
    df_from_database = pd.DataFrame(
        rows, columns=["column1", "text_chunks", "column3"]
    )
    text_chunks_existing = df_from_database["text_chunks"]

    # compare text_chunks and text_chunks_existing and only keep the ones which are not already in the table
    text_chunks = [
        text_chunk
        for text_chunk in text_chunks
        if text_chunk not in text_chunks_existing
    ]

except:
    pass

client = OpenAI()
model = "text-embedding-3-small"

print(len(text_chunks))

# Insert the embeddings into the table using OpenAI API
for text_chunk in text_chunks:
    print(text_chunk)
    response = client.embeddings.create(input=text_chunk, model=model)
    embedding = response.data[0].embedding

    # assign a new id to the text chunk
    cur.execute("SELECT MAX(id) FROM job_description_table")
    max_id = cur.fetchone()[0]
    index = max_id + 1 if max_id else 1

    # Insert the id, content, and embedding into the table
    cur.execute(
        "INSERT INTO job_description_table (id, text_chunk, embedding) VALUES (%s, %s, %s)",
        (index, text_chunk, embedding),
    )
    conn.commit()
    print(index)

In [ ]:
results

### 6 Perform a vector search using HNSW and IVFLL index

In [ ]:
query = "I am looking for a job as a data scientist in Berlin."
client = OpenAI()
model = "text-embedding-3-small"
response = client.embeddings.create(input=query, model=model)
query_embedding = response.data[0].embedding

full_search_sql = f"""
    EXPLAIN ANALYZE SELECT 1 - (embedding <=> '{str(query_embedding)}')
    AS cosine_similarity,
    * FROM job_description_table
    ORDER BY 1 - (embedding <=> '{str(query_embedding)}') DESC
    LIMIT 20;
    """

cur.execute(full_search_sql)

results_full_search = cur.fetchall()
# [..., ('Planning Time: 13.669 ms',), ('Execution Time: 85.582 ms',)]

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
import psycopg2
from psycopg2 import Error

ivfflat_sql = f"""
    DROP TABLE IF EXISTS test_embedding_table;
    CREATE TABLE test_embedding_table AS
        SELECT * FROM job_description_table;
    CREATE INDEX ON test_embedding_table
        USING ivfflat (embedding vector_cosine_ops)
        WITH (lists = 30);
    -- Reduce the number of probes for faster search
    SET ivfflat.probes = 3;
    EXPLAIN ANALYZE SELECT 1 - (embedding <=> '{str(query_embedding)}')
    AS cosine_similarity, *
    FROM test_embedding_table
    ORDER BY 1 - (embedding <=> '{str(query_embedding)}') DESC
    LIMIT 20;
    """

cur.execute(ivfflat_sql)
ivfflat_search = cur.fetchall()

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
import psycopg2
from psycopg2 import Error

hnsw_test_sql = f"""
    DROP TABLE IF EXISTS test_embedding_table;
    CREATE TABLE test_embedding_table AS
        SELECT * FROM job_description_table;
    CREATE INDEX ON test_embedding_table
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 200);
    SET hnsw.ef_search = 50;
    EXPLAIN ANALYZE SELECT 1 - (embedding <=> '{str(query_embedding)}')
    AS cosine_similarity, *
    FROM test_embedding_table
    ORDER BY 1 - (embedding <=> '{str(query_embedding)}') DESC
    LIMIT 20;
"""

cur.execute(hnsw_test_sql)
hnsw_search = cur.fetchall()

# [..., ('Planning Time: 1.109 ms',), ('Execution Time: 13.927 ms',)]

In [ ]:
ivfflat_search, hnsw_search, results_full_search = perform_hnsw_similarity_search()
print(f"Index HSNW search: {hnsw_search[6]}")
print(f"Index IVFFLat search: {ivfflat_search[6]}")
print(f"Full search: {results_full_search[6]}")


### 7 Hybrid search using PostgreSQL

In [ ]:
import psycopg2
from psycopg2 import Error

conn = psycopg2.connect(
    dbname="rag_cookbook",
    user="rag_cookbook_user",
    password="rag_cookbook_user_pw",
    host="localhost",
    port="5432",
)

cur = conn.cursor()

hybrid_search_table = f"""
    DROP TABLE IF EXISTS test_embedding_table;
    CREATE TABLE test_embedding_table AS
    SELECT *, to_tsvector(text_chunk) AS tsv
    FROM job_description_table;
    """

cur.execute(hybrid_search_table)

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
import psycopg2
from psycopg2 import Error
from openai import OpenAI

query = "I am looking for a job as a data scientist in Berlin."
client = OpenAI()
model = "text-embedding-3-small"
response = client.embeddings.create(input=query, model=model)
query_embedding = response.data[0].embedding

hybrid_search_query = f"""
    WITH ranked_docs AS (
        SELECT
            id,
            text_chunk AS text,
            ts_rank(tsv, plainto_tsquery('PostgreSQL')) AS text_score,
            1 - (embedding <=> '{str(query_embedding)}') AS vector_score,
            ts_rank(tsv, plainto_tsquery('PostgreSQL'))
            * 0.5 + (1 - (embedding <=> '{str(query_embedding)}')) * 0.5
            AS hybrid_score
        FROM test_embedding_table
        -- Filter for relevant text first
        WHERE tsv @@ plainto_tsquery('PostgreSQL')
        ORDER BY hybrid_score DESC
        LIMIT 10
    )
    SELECT * FROM ranked_docs;
    """

cur.execute(hybrid_search_query)
results = cur.fetchall()